Mean Variance Portfolio Optimization (Markowitz)

1 Imports

2 Define Assets

3 Download Data

4 Returns Matrix

5 Expected Returns

6 Covariance Matrix

7 Portfolio Functions

8 Minimum Variance Optimization

9 Portfolio Metrics

10 Random Portfolio Simulation

11 Efficient Frontier

12 Tangency Portfolio

13 Best Random Portfolio

14 Capital Market Line

15 Final Visualization

16 Portfolio Weights

17 Out-of-Sample Backtest

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy.optimize import minimize
plt.style.use("dark_background")

In [ ]:
stocks = [
    "MODEFENCE.NS",
    "CUMMINSIND.NS",
    "POLYCAB.NS",
    "M&M.NS",
    "INDIANB.NS",
    "MON100.NS"
]
data = yf.download(
    stocks,
    start="2022-01-01",
    auto_adjust=True
)["Close"]

print(data.head())
print(data.columns)

In [ ]:
returns = data.pct_change().dropna()
returns.head()

In [ ]:
mu = returns.mean() * 252
mu

In [ ]:
cov_matrix = returns.cov() * 252
cov_matrix

In [ ]:
def portfolio_return(w, mu):
    return np.dot(w, mu)

def portfolio_variance(w, cov):
    return w.T @ cov @ w

In [ ]:
n = len(stocks)

def portfolio_var(w):
    return w.T @ cov_matrix @ w

constraints = (
    {"type": "eq", "fun": lambda w: np.sum(w) - 1},
)

bounds = tuple((0,1) for _ in range(n))

w0 = np.ones(n) / n

result = minimize(
    portfolio_var,
    w0,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

optimal_weights = result.x

In [ ]:
min_var_return = np.dot(optimal_weights, mu)
min_var_vol = np.sqrt(optimal_weights.T @ cov_matrix @ optimal_weights)

print("Minimum Variance Portfolio")
print("Return:", min_var_return)
print("Volatility:", min_var_vol)

In [ ]:
num_portfolios = 10000

returns_list = []
risk_list = []
sharpe_list = []
weights_list = []

rf = 0.05

for i in range(num_portfolios):

    w = np.random.random(n)
    w /= np.sum(w)

    ret = np.dot(w, mu)
    risk = np.sqrt(w.T @ cov_matrix @ w)

    sharpe = (ret - rf) / risk

    returns_list.append(ret)
    risk_list.append(risk)
    sharpe_list.append(sharpe)
    weights_list.append(w)

In [ ]:
target_returns = np.linspace(mu.min(), mu.max(), 50)

frontier_risk = []

for target in target_returns:

    constraints = (
        {"type":"eq","fun":lambda w: np.sum(w)-1},
        {"type":"eq","fun":lambda w: np.dot(w,mu)-target}
    )

    result = minimize(
        portfolio_var,
        w0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints
    )

    frontier_risk.append(np.sqrt(result.fun))

In [ ]:
def negative_sharpe(w):

    ret = np.dot(w, mu)
    vol = np.sqrt(w.T @ cov_matrix @ w)

    return -(ret - rf) / vol

constraints = (
    {"type":"eq","fun":lambda w: np.sum(w)-1},
)

result = minimize(
    negative_sharpe,
    w0,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints
)

sharpe_weights = result.x

In [ ]:
tangency_return = np.dot(sharpe_weights, mu)
tangency_vol = np.sqrt(sharpe_weights.T @ cov_matrix @ sharpe_weights)

print("Tangency Portfolio")
print("Return:", tangency_return)
print("Volatility:", tangency_vol)

In [ ]:
max_sharpe_idx = np.argmax(sharpe_list)

best_random_return = returns_list[max_sharpe_idx]
best_random_risk = risk_list[max_sharpe_idx]

In [ ]:
cml_x = np.linspace(0, max(risk_list), 100)

cml_y = rf + (tangency_return - rf) / tangency_vol * cml_x

In [ ]:
plt.figure(figsize=(12,7))

scatter = plt.scatter(
    risk_list,
    returns_list,
    c=sharpe_list,
    cmap="viridis",
    alpha=0.35
)

plt.colorbar(scatter,label="Sharpe Ratio")

plt.plot(
    frontier_risk,
    target_returns,
    color="red",
    linewidth=3,
    label="Efficient Frontier"
)

plt.scatter(
    min_var_vol,
    min_var_return,
    color="blue",
    marker="*",
    s=300,
    label="Minimum Variance Portfolio"
)

plt.scatter(
    tangency_vol,
    tangency_return,
    color="green",
    marker="*",
    s=300,
    label="Tangency Portfolio"
)

plt.scatter(
    best_random_risk,
    best_random_return,
    color="orange",
    marker="*",
    s=300,
    label="Best Random Portfolio"
)

plt.plot(
    cml_x,
    cml_y,
    color="white",
    linestyle="--",
    linewidth=2,
    label="Capital Market Line"
)

plt.xlabel("Volatility (Risk)")
plt.ylabel("Expected Return")
plt.title("Efficient Frontier, Optimal Portfolio & Capital Market Line")
plt.legend()
plt.savefig("efficient_frontier.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
print("Minimum Variance Portfolio Weights\n")

for stock, weight in zip(stocks, optimal_weights):
    print(f"{stock}: {weight:.3f}")

In [ ]:
print("\nTangency Portfolio Weights\n")

for stock, weight in zip(stocks, sharpe_weights):
    print(f"{stock}: {weight:.3f}")

In [ ]:
train_returns = returns.loc["2022":"2023"]
test_returns = returns.loc["2024":]

In [ ]:
mu_train = train_returns.mean() * 252
cov_train = train_returns.cov() * 252

In [ ]:
portfolio_test_returns = test_returns @ optimal_weights

equal_weights = np.ones(n) / n
equal_test_returns = test_returns @ equal_weights

In [ ]:
portfolio_cum = (1 + portfolio_test_returns).cumprod()
equal_cum = (1 + equal_test_returns).cumprod()

In [ ]:
plt.figure(figsize=(10,6))

plt.plot(portfolio_cum,label="Optimized Portfolio")
plt.plot(equal_cum,label="Equal Weight Portfolio")

plt.title("Out-of-Sample Performance")

plt.xlabel("Date")
plt.ylabel("Growth of $1")

plt.legend()

plt.show()